In [3]:
%pip install geopandas

from pathlib import Path
import pandas as pd
import geopandas as gpd

# 1. read geojson
park_path = Path("VPA_Draft_Open_Space_Data_-7361858239455755329.geojson")
park_gdf = gpd.read_file(park_path)

print("Raw shape:", park_gdf.shape)
print("CRS:", park_gdf.crs)
print("Columns:")
print(park_gdf.columns.tolist())

Note: you may need to restart the kernel to use updated packages.


f:\anaconda\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


Raw shape: (38810, 23)
CRS: EPSG:3857
Columns:
['FID', 'LGA', 'VM_PARCEL_', 'VM_PARCE_1', 'DATA_SOURC', 'OS_CATEGOR', 'OS_CATEG_2', 'OWNER_TYPE', 'PARK_NAME', 'OS_STATUS', 'OS_ACCESS', 'MANAGER_TY', 'HA', 'SUBREGION', 'VEAC_ID', 'WATER_BODY', 'OS_TYPE', 'COASTAL', 'MANAGER_NA', 'OWNER_NAME', 'Image_URL', 'VPA_ID', 'geometry']


In [4]:
# 2. keep useful columns
PARK_KEEP_COLS = [
    "FID",
    "LGA",
    "OS_CATEGOR",
    "OS_CATEG_2",
    "OWNER_TYPE",
    "PARK_NAME",
    "OS_STATUS",
    "OS_ACCESS",
    "MANAGER_TY",
    "HA",
    "SUBREGION",
    "OS_TYPE",
    "MANAGER_NA",
    "OWNER_NAME",
    "VPA_ID",
    "geometry",
]

park_gdf = park_gdf[PARK_KEEP_COLS].copy()
print("After column selection:", park_gdf.shape)

After column selection: (38810, 16)


In [5]:
# 3. text cleaning
TEXT_COLS = [
    "LGA",
    "OS_CATEGOR",
    "OS_CATEG_2",
    "OWNER_TYPE",
    "PARK_NAME",
    "OS_STATUS",
    "OS_ACCESS",
    "MANAGER_TY",
    "SUBREGION",
    "OS_TYPE",
    "MANAGER_NA",
    "OWNER_NAME",
]

def normalize_text(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip()
    if value == "":
        return pd.NA
    value = " ".join(value.split())
    return value

for col in TEXT_COLS:
    park_gdf[col] = park_gdf[col].apply(normalize_text)

park_gdf["HA"] = pd.to_numeric(park_gdf["HA"], errors="coerce")

In [6]:
# 4. filter Melbourne city parks
melbourne_parks = park_gdf[
    park_gdf["LGA"].astype("string").str.upper().eq("MELBOURNE")
].copy()

melbourne_parks = melbourne_parks[
    melbourne_parks["OS_CATEGOR"].astype("string").str.lower().eq("parks and gardens")
].copy()

melbourne_parks = melbourne_parks[
    melbourne_parks["OS_TYPE"].astype("string").str.lower().eq("public open space")
].copy()

melbourne_parks = melbourne_parks[
    melbourne_parks["OS_ACCESS"].astype("string").str.lower().eq("open")
].copy()

print("After park filter:", melbourne_parks.shape)
print("Unique park names:", melbourne_parks["PARK_NAME"].nunique(dropna=True))

After park filter: (299, 16)
Unique park names: 138


In [7]:
# 5. remove invalid park names
invalid_park_names = {
    "NO DATA", "N/A", "NA", "NULL", "NONE", "-", ".", "...", ""
}

melbourne_parks["PARK_NAME"] = melbourne_parks["PARK_NAME"].astype("string").str.strip()

melbourne_parks = melbourne_parks[
    melbourne_parks["PARK_NAME"].notna()
].copy()

melbourne_parks = melbourne_parks[
    ~melbourne_parks["PARK_NAME"].str.upper().isin(invalid_park_names)
].copy()

print("After PARK_NAME cleaning:", melbourne_parks.shape)
print("Unique valid park names:", melbourne_parks["PARK_NAME"].nunique(dropna=True))

After PARK_NAME cleaning: (290, 16)
Unique valid park names: 137


In [23]:
# 6. dissolve same park name into one geometry
park_dissolved = melbourne_parks.dissolve(
    by="PARK_NAME",
    aggfunc={
        "LGA": "first",
        "OS_CATEGOR": "first",
        "OS_CATEG_2": "first",
        "OWNER_TYPE": "first",
        "OS_STATUS": "first",
        "OS_ACCESS": "first",
        "MANAGER_TY": "first",
        "HA": "sum",
        "SUBREGION": "first",
        "OS_TYPE": "first",
        "MANAGER_NA": "first",
        "OWNER_NAME": "first",
        "VPA_ID": "first",
    }
).reset_index()

print("After dissolve:", park_dissolved.shape)
park_dissolved.head()

After dissolve: (137, 15)


,PARK_NAME,geometry,LGA,OS_CATEGOR,OS_CATEG_2,OWNER_TYPE,OS_STATUS,OS_ACCESS,MANAGER_TY,HA,SUBREGION,OS_TYPE,MANAGER_NA,OWNER_NAME,VPA_ID
0,Alexandra Gardens,"MULTIPOLYGON (((1.61e+07 -4.56e+06, 1.61e+07 -...",MELBOURNE,Parks and gardens,Not applicable,Crown,Existing,Open,NO DATA,5.7493,Central,Public open space,NO DATA,Crown,35800
1,Alexandra Park,"MULTIPOLYGON (((1.61e+07 -4.55e+06, 1.61e+07 -...",MELBOURNE,Parks and gardens,Metropolitan link,Crown,Existing,Open,NO DATA,4.8691,Central,Public open space,NO DATA,Crown,35829
2,Argyle Square,"POLYGON ((1.61e+07 -4.55e+06, 1.61e+07 -4.55e+...",MELBOURNE,Parks and gardens,Not applicable,Crown,Existing,Open,NO DATA,1.3297,Central,Public open space,NO DATA,Crown,35221
3,Auckland Lane Reserve,"POLYGON ((1.61e+07 -4.55e+06, 1.61e+07 -4.55e+...",MELBOURNE,Parks and gardens,Not applicable,Crown,Existing,Open,NO DATA,0.0323,Central,Public open space,NO DATA,Crown,35707
4,Batman Park,"POLYGON ((1.61e+07 -4.55e+06, 1.61e+07 -4.55e+...",MELBOURNE,Parks and gardens,Not applicable,Crown,Existing,Open,NO DATA,1.4328,Central,Public open space,NO DATA,Crown,35692


In [24]:
# Keep a polygon version for later spatial join
park_polygon_clean = park_dissolved.copy()

# Rename fields to match our project style
park_polygon_clean = park_polygon_clean.rename(columns={
    "PARK_NAME": "park_name",
    "HA": "ha",
})

# Add park_id, same logic as final CSV
park_polygon_clean.insert(
    0,
    "park_id",
    range(1, len(park_polygon_clean) + 1)
)

# Export polygon GeoJSON for future spatial join
park_polygon_output = Path("park_polygon_clean.geojson")
park_polygon_clean.to_file(park_polygon_output, driver="GeoJSON")

print("Exported polygon file:", park_polygon_output)
print("Polygon row count:", len(park_polygon_clean))
print("CRS:", park_polygon_clean.crs)

Exported polygon file: park_polygon_clean.geojson
Polygon row count: 137
CRS: EPSG:3857


In [9]:
# 7. calculate centroid in projected CRS
park_dissolved["centroid_geom"] = park_dissolved.geometry.centroid

# if later you find centroid not ideal, replace with:
# park_dissolved["centroid_geom"] = park_dissolved.geometry.representative_point()

park_centroids = gpd.GeoDataFrame(
    park_dissolved.drop(columns="geometry"),
    geometry=park_dissolved["centroid_geom"],
    crs=park_dissolved.crs,
).drop(columns="centroid_geom")

print("Centroid CRS:", park_centroids.crs)
park_centroids.head()

Centroid CRS: EPSG:3857


,PARK_NAME,LGA,OS_CATEGOR,OS_CATEG_2,OWNER_TYPE,OS_STATUS,OS_ACCESS,MANAGER_TY,HA,SUBREGION,OS_TYPE,MANAGER_NA,OWNER_NAME,VPA_ID,geometry
0,Alexandra Gardens,MELBOURNE,Parks and gardens,Not applicable,Crown,Existing,Open,NO DATA,5.7493,Central,Public open space,NO DATA,Crown,35800,POINT (1.61e+07 -4.55e+06)
1,Alexandra Park,MELBOURNE,Parks and gardens,Metropolitan link,Crown,Existing,Open,NO DATA,4.8691,Central,Public open space,NO DATA,Crown,35829,POINT (1.61e+07 -4.55e+06)
2,Argyle Square,MELBOURNE,Parks and gardens,Not applicable,Crown,Existing,Open,NO DATA,1.3297,Central,Public open space,NO DATA,Crown,35221,POINT (1.61e+07 -4.55e+06)
3,Auckland Lane Reserve,MELBOURNE,Parks and gardens,Not applicable,Crown,Existing,Open,NO DATA,0.0323,Central,Public open space,NO DATA,Crown,35707,POINT (1.61e+07 -4.55e+06)
4,Batman Park,MELBOURNE,Parks and gardens,Not applicable,Crown,Existing,Open,NO DATA,1.4328,Central,Public open space,NO DATA,Crown,35692,POINT (1.61e+07 -4.55e+06)


In [10]:
# 8. convert centroid to WGS84 lat/lon
park_centroids_wgs84 = park_centroids.to_crs(epsg=4326).copy()

park_centroids_wgs84["longitude"] = park_centroids_wgs84.geometry.x
park_centroids_wgs84["latitude"] = park_centroids_wgs84.geometry.y

print(park_centroids_wgs84[["PARK_NAME", "latitude", "longitude"]].head())

               PARK_NAME   latitude   longitude
0      Alexandra Gardens -37.821402  144.973383
1         Alexandra Park -37.824079  144.977433
2          Argyle Square -37.802740  144.965835
3  Auckland Lane Reserve -37.777987  144.938940
4            Batman Park -37.821785  144.956613


In [12]:
park_centroids_wgs84["location_label"] = "MELBOURNE"

def build_st_point(longitude, latitude):
    if pd.isna(longitude) or pd.isna(latitude):
        return pd.NA
    return f"POINT({longitude} {latitude})"

park_centroids_wgs84["st_point"] = park_centroids_wgs84.apply(
    lambda row: build_st_point(row["longitude"], row["latitude"]),
    axis=1
)

park_centroids_wgs84["is_active"] = True

In [20]:
# build st_point
def build_st_point(longitude, latitude):
    if pd.isna(longitude) or pd.isna(latitude):
        return pd.NA
    return f"POINT({longitude} {latitude})"

park_centroids_wgs84["st_point"] = park_centroids_wgs84.apply(
    lambda row: build_st_point(row["longitude"], row["latitude"]),
    axis=1
)

# map to database fields
park_centroids_wgs84["park_name"] = park_centroids_wgs84["PARK_NAME"]
park_centroids_wgs84["ha"] = park_centroids_wgs84["HA"]
park_centroids_wgs84["description"] = park_centroids_wgs84["park_name"].apply(
    lambda x: f"{x} is a public park in Melbourne." if pd.notna(x) else pd.NA
)
park_centroids_wgs84["description"] = park_centroids_wgs84.apply(
    lambda row: f"{row['park_name']} is a public park in {row['location_label'].title()}, Melbourne."
    if pd.notna(row["park_name"]) and pd.notna(row["location_label"]) 
    else f"{row['park_name']} is a public park in Melbourne." if pd.notna(row["park_name"]) 
    else pd.NA,
    axis=1
)
# final table
park_final = park_centroids_wgs84[
    ["park_name", "ha","latitude", "longitude", "st_point", "description"]
].copy()

# add primary key
park_final.insert(0, "park_id", range(1, len(park_final) + 1))

# export
output_path = Path("melbourne_city_parks_final.csv")
park_final.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Exported to:", output_path)
print("Row count:", len(park_final))
print("Column count:", len(park_final.columns))
print(park_final.head())

Exported to: melbourne_city_parks_final.csv
Row count: 137
Column count: 7
   park_id              park_name      ha   latitude   longitude  \
0        1      Alexandra Gardens  5.7493 -37.821402  144.973383   
1        2         Alexandra Park  4.8691 -37.824079  144.977433   
2        3          Argyle Square  1.3297 -37.802740  144.965835   
3        4  Auckland Lane Reserve  0.0323 -37.777987  144.938940   
4        5            Batman Park  1.4328 -37.821785  144.956613   

                                        st_point  \
0   POINT(144.97338326664902 -37.82140175342995)   
1   POINT(144.9774328963564 -37.824079314048966)   
2  POINT(144.96583511130225 -37.802740458199075)   
3    POINT(144.9389399141761 -37.77798700546881)   
4    POINT(144.9566130685186 -37.82178529606905)   

                                         description  
0  Alexandra Gardens is a public park in Melbourn...  
1  Alexandra Park is a public park in Melbourne, ...  
2  Argyle Square is a public park in M

In [21]:
park_final.head(5)

,park_id,park_name,ha,latitude,longitude,st_point,description
0,1,Alexandra Gardens,5.7493,-37.821402,144.973383,POINT(144.97338326664902 -37.82140175342995),Alexandra Gardens is a public park in Melbourn...
1,2,Alexandra Park,4.8691,-37.824079,144.977433,POINT(144.9774328963564 -37.824079314048966),"Alexandra Park is a public park in Melbourne, ..."
2,3,Argyle Square,1.3297,-37.802740,144.965835,POINT(144.96583511130225 -37.802740458199075),"Argyle Square is a public park in Melbourne, M..."
3,4,Auckland Lane Reserve,0.0323,-37.777987,144.938940,POINT(144.9389399141761 -37.77798700546881),Auckland Lane Reserve is a public park in Melb...
4,5,Batman Park,1.4328,-37.821785,144.956613,POINT(144.9566130685186 -37.82178529606905),"Batman Park is a public park in Melbourne, Mel..."


In [27]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

park_polygon_path = Path("park_polygon_clean.geojson")
artwork_path = Path("tasks_ready_artworks.csv")
landmark_path = Path("tasks_ready_landmarks.csv")

park_gdf = gpd.read_file(park_polygon_path)
artwork_df = pd.read_csv(artwork_path)
landmark_df = pd.read_csv(landmark_path)

print("Park polygon shape:", park_gdf.shape)
print("Park CRS:", park_gdf.crs)

print("Artwork shape:", artwork_df.shape)
print("Landmark shape:", landmark_df.shape)

Park polygon shape: (137, 16)
Park CRS: EPSG:3857
Artwork shape: (164, 14)
Landmark shape: (131, 14)


In [28]:
artwork_feature = artwork_df.copy()
artwork_feature["feature_source"] = "artwork"
artwork_feature["feature_name"] = artwork_feature["task_name"]
artwork_feature["feature_category"] = "art"

landmark_feature = landmark_df.copy()
landmark_feature["feature_source"] = "landmark"
landmark_feature["feature_name"] = landmark_feature["task_name"]

landmark_feature["series_id"] = pd.to_numeric(
    landmark_feature["series_id"],
    errors="coerce"
).astype("Int64")

def classify_landmark_category(series_id):
    if series_id == 1:
        return "nature"
    elif series_id == 2:
        return "urban"
    elif series_id == 3:
        return "art"
    else:
        return "urban"

landmark_feature["feature_category"] = landmark_feature["series_id"].apply(
    classify_landmark_category
)

feature_points = pd.concat(
    [artwork_feature, landmark_feature],
    ignore_index=True
)

print("Combined feature points:", feature_points.shape)
print(feature_points["feature_source"].value_counts(dropna=False))
print(feature_points["feature_category"].value_counts(dropna=False))

Combined feature points: (295, 17)
feature_source
artwork     164
landmark    131
Name: count, dtype: int64
feature_category
art       200
urban      52
nature     43
Name: count, dtype: int64


In [29]:
feature_points["latitude"] = pd.to_numeric(feature_points["latitude"], errors="coerce")
feature_points["longitude"] = pd.to_numeric(feature_points["longitude"], errors="coerce")

feature_gdf = gpd.GeoDataFrame(
    feature_points,
    geometry=gpd.points_from_xy(
        feature_points["longitude"],
        feature_points["latitude"]
    ),
    crs="EPSG:4326"
)

feature_gdf = feature_gdf.to_crs(park_gdf.crs)

print("Feature CRS after transform:", feature_gdf.crs)
print("Feature shape:", feature_gdf.shape)

Feature CRS after transform: EPSG:3857
Feature shape: (295, 18)


In [30]:
# Keep only key park columns before join
park_join_cols = [
    "park_id",
    "park_name",
    "geometry"
]

park_for_join = park_gdf[park_join_cols].copy()

# Spatial join: feature point inside park polygon
feature_joined = gpd.sjoin(
    feature_gdf,
    park_for_join,
    how="inner",
    predicate="within"
)

print("Joined feature count:", feature_joined.shape)
print("Matched parks:", feature_joined["park_name"].nunique(dropna=True))
print(feature_joined[["feature_source", "feature_category", "feature_name", "park_name"]].head())

Joined feature count: (121, 21)
Matched parks: 42
  feature_source feature_category                                feature_name  \
3        artwork              art                            Federation Bells   
4        artwork              art  Thomas Ferguson Memorial Drinking Fountain   
5        artwork              art                             The Water Nymph   
6        artwork              art                                    Blowhole   
7        artwork              art                          Mockridge Fountain   

                                  park_name  
3                            Birrarung Marr  
4                         University Square  
5  Queen Victoria Gardens & Memorial Statue  
6                            Docklands Park  
7                               City Square  


In [31]:
print("Feature source count:")
print(feature_joined["feature_source"].value_counts(dropna=False))

print("\nFeature category count:")
print(feature_joined["feature_category"].value_counts(dropna=False))

Feature source count:
feature_source
artwork     86
landmark    35
Name: count, dtype: int64

Feature category count:
feature_category
art       89
nature    29
urban      3
Name: count, dtype: int64


In [32]:
feature_joined_out = feature_joined[
    [
        "park_id",
        "park_name",
        "feature_source",
        "feature_name",
        "feature_category",
        "series_id",
        "latitude",
        "longitude",
    ]
].copy()

output_path = Path("park_feature_join_artwork_landmark.csv")
feature_joined_out.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Exported:", output_path)
print("Rows:", len(feature_joined_out))

feature_joined_out = feature_joined_out.drop_duplicates(
    subset=[
        "park_id",
        "feature_source",
        "feature_name",
        "feature_category",
        "latitude",
        "longitude",
    ]
).copy()

print("Rows after dedup:", len(feature_joined_out))

Exported: park_feature_join_artwork_landmark.csv
Rows: 121
Rows after dedup: 121
